In [108]:
import pandas as pd
import altair as alt
from vega_datasets import data

In [109]:
gen_df = pd.read_csv('presidents_general.csv')
vote_df = pd.read_csv('electoral_data.csv')
df = pd.read_csv('merged_presidents.csv')
fips = pd.read_csv('us-state-ansi-fips.csv')

In [110]:
# filtering vote data to >= 1865, 1860 was the last election to have more than 3 parties on the ballot (1865 b/c Johnson took office after Lincoln's assassination)
vote_df = vote_df[vote_df['Year'] >= 1864]
df = df[df['Year'] > 1865]

In [111]:
print("Most common birth state:", gen_df['Birth State'].mode(dropna=True)[0])
print("Most common death state:", gen_df['Death State'].mode(dropna=True)[0])
print("Most common university attended:",
      gen_df['University'].mode(dropna=True)[0])
print("Most common degree held:",
      gen_df['Highest Degree Held'].mode(dropna=True)[0])
print("Average number of lanugages spoken:",
      gen_df['Languages'].mean())
print("Average number of countries visited as president:",
      gen_df['# Countries Visited as President'].mean())
print("Average % of electoral votes recieved:",
      vote_df['Electoral College'].mean())
print("Average % of popular vote recieved:",
      vote_df['Popular Vote'].mean())

Most common birth state: Virginia
Most common death state: New York
Most common university attended: Harvard University 
Most common degree held: Bachelors
Average number of lanugages spoken: 1.9
Average number of countries visited as president: 25.523809523809526
Average % of electoral votes recieved: 70.75
Average % of popular vote recieved: 52.017073170731706


In [112]:
birth_counts = (
    gen_df
    .groupby('Birth State')
    .size()
    .reset_index(name='count')
)

death_counts = (
    gen_df
    .groupby('Death State')
    .size()
    .reset_index(name='count')
)

# merging data w/ fips csv to map for altair lookup, from https://gist.github.com/dantonnoriega/bf1acd2290e15b91e6710b6fd3be0a53

birth_counts = pd.merge(
    birth_counts, fips[['stname', 'st']], left_on='Birth State', right_on='stname', how='left')
death_counts = pd.merge(
    death_counts, fips[['stname', 'st']], left_on='Death State', right_on='stname', how='left')

In [133]:
usa = data.us_10m.url

birth = alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
    stroke='#aaa', strokeWidth=0.25
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data=birth_counts, key='st',
                         fields=['count', 'stname']),
    default='0'
).encode(
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='blues'),
        legend=alt.Legend(title=None)
    ),
    tooltip=[
        alt.Tooltip('count:Q', title='# Presidents Born'),
        alt.Tooltip('stname:N', title='State')
    ]
).project(
    type='albersUsa'
).properties(
    width=600, height=500)

death = alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
    stroke='#aaa', strokeWidth=0.25
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(data=death_counts, key='st',
                         fields=['count', 'stname']),
    default='0'
).encode(
    color=alt.Color(
        'count:Q',
        scale=alt.Scale(scheme='blues'),
        legend=alt.Legend(title='# Presidents Dead')
    ),
    tooltip=[
        alt.Tooltip('count:Q', title='# Presidents Died'),
        alt.Tooltip('stname:N', title='State'),

    ]
).project(
    type='albersUsa'
).properties(
    width=600, height=500)

(birth | death).properties(title='US Presidents by Birth and Death States ').configure_view(
    stroke=None
)

alt.HConcatChart(...)